In [ ]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

sys.path.append(str(SRC_DIR))

from paperscope.models import Chunk
from paperscope.retrieval import reranked_retrieve
from paperscope.retrieval import smart_retrieve

In [ ]:
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.json"

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunk_data = json.load(f)

all_chunks = [Chunk(**item) for item in chunk_data]

print("Loaded chunks:", len(all_chunks))

In [ ]:
from qdrant_client import QdrantClient

QDRANT_PATH = PROJECT_ROOT / "storage" / "qdrant"
COLLECTION_NAME = "research_papers"

client = QdrantClient(
    path=str(QDRANT_PATH)
)

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [
    chunk.text.lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

embedding_model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cuda"
)

reranker = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B",
    device="cuda"
)

In [ ]:
query = "How does Self-RAG decide when retrieval is necessary?"

results = reranked_retrieve(
    query=query,
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    bm25=bm25,
    chunks=all_chunks,
    reranker=reranker,
    candidate_k=15,
    top_k=5
)

In [ ]:
for rank, result in enumerate(results, start=1):
    chunk = result["chunk"]

    print(
        rank,
        round(result["rerank_score"], 4),
        "|",
        chunk["title"],
        "|",
        chunk["section_heading"]
    )

In [ ]:
context_parts = []

for i, result in enumerate(results, start=1):
    chunk = result["chunk"]

    context_parts.append(
        f"""
[SOURCE {i}]
Paper: {chunk['title']}
Section: {chunk['section_heading']}
Pages: {chunk['page_start']}-{chunk['page_end']}

{chunk['text']}
"""
    )

context = "\n".join(context_parts)

In [ ]:
prompt = f"""
You are a research assistant answering questions about a collection of research papers.

Use only the evidence provided below.

Rules:
- Do not use outside knowledge.
- Every factual claim should be supported by the supplied evidence.
- Cite sources using [SOURCE 1], [SOURCE 2], etc.
- Prefer primary-source evidence over papers that merely discuss another work.
- If the evidence is insufficient, explicitly say that the provided papers do not contain enough information.
- Do not invent citations.

Question:
{query}

Evidence:
{context}

Answer:
"""

In [ ]:
print(prompt[:6000])

In [ ]:
import importlib

importlib.reload(paperscope.generation)
importlib.reload(paperscope.retrieval)
import paperscope.generation
from paperscope.generation import generate_answer

# answer = generate_answer(
#     query=query,
#     results=results
# )

# print(answer)

In [ ]:
query = "Compare the RAG survey and CRAG."

results = smart_retrieve(
    query=query,
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    bm25=bm25,
    chunks=all_chunks,
    reranker=reranker,
    top_k=5
)

In [ ]:
from paperscope.generation import generate_answer

answer = generate_answer(
    query=query,
    results=results
)

print(answer)